In [ ]:
# ============================================
# IMPORT THÊM THƯ VIỆN
# ============================================

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

# ============================================
# LƯU KẾT QUẢ CÁC MODEL
# ============================================

results = {}

# Machine Learning models
for name, model in models.items():

    y_probs = model.predict_proba(X_test_ga)[:, 1]
    y_pred = (y_probs > threshold).astype(int)

    results[name] = {
        "y_pred": y_pred,
        "y_probs": y_probs
    }

# Neural Network
nn_probs = nn_model.predict(X_test_ga).ravel()
nn_preds = (nn_probs > threshold).astype(int)

results["Neural Network"] = {
    "y_pred": nn_preds,
    "y_probs": nn_probs
}
# ============================================
# CONFUSION MATRIX
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (name, result) in enumerate(results.items()):

    cm = confusion_matrix(y_test, result["y_pred"])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Stroke", "Stroke"]
    )

    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)

    axes[idx].set_title(f"{name}")

plt.tight_layout()
plt.show()
# ============================================
# ROC CURVE + AUC
# ============================================

plt.figure(figsize=(10, 8))

auc_scores = {}

for name, result in results.items():

    fpr, tpr, _ = roc_curve(y_test, result["y_probs"])

    roc_auc = auc(fpr, tpr)

    auc_scores[name] = roc_auc

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{name} (AUC = {roc_auc:.4f})"
    )

# Đường random
plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)

plt.show()
# ============================================
# HIỂN THỊ AUC
# ============================================

print("\n========== AUC SCORES ==========")

for name, score in auc_scores.items():
    print(f"{name}: {score:.4f}")
    # ============================================
# ERROR ANALYSIS
# ============================================

best_model_name = "Decision Tree"

y_pred_best = results[best_model_name]["y_pred"]

# False Positive
fp_idx = np.where((y_test == 0) & (y_pred_best == 1))[0]

# False Negative
fn_idx = np.where((y_test == 1) & (y_pred_best == 0))[0]

print(f"\nFalse Positives: {len(fp_idx)}")
print(f"False Negatives: {len(fn_idx)}")
# Convert lại DataFrame để dễ xem
X_test_df = pd.DataFrame(
    X_test_ga,
    columns=selected_feature_names
)

# False Negative samples
print("\n===== FALSE NEGATIVE SAMPLES =====")
display(X_test_df.iloc[fn_idx[:5]])

# False Positive samples
print("\n===== FALSE POSITIVE SAMPLES =====")
display(X_test_df.iloc[fp_idx[:5]])
# ============================================
# FEATURE IMPORTANCE
# ============================================

dt_model = models["Decision Tree"]

importances = dt_model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": selected_feature_names,
    "Importance": importances
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance_df.head(10))
# ============================================
# FEATURE IMPORTANCE PLOT
# ============================================

plt.figure(figsize=(10, 6))

sns.barplot(
    x="Importance",
    y="Feature",
    data=feature_importance_df.head(10)
)

plt.title("Top 10 Most Important Features")
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.show()

NameError: name 'X_train_ga' is not defined